## Deliverable 2

### Jon Gundesø og Jonas Kristoffersen

### Task 1

### Task 2

In [1]:
import json
from pathlib import Path


def find_log_files(path: str | Path) -> list[Path]:
    """Find all .log files in a directory and its subdirectories."""
    path = Path(path)
    log_files = []

    for item in path.iterdir():
        if item.is_dir():
            log_files.extend(find_log_files(item))
        elif item.is_file() and item.suffix == ".log":
            log_files.append(item)

    return log_files


def find_activeID(data: dict | list) -> set[str]:
    """Find all register IDs where active is True."""

    active_registers = set()

    if isinstance(data, dict):
        if data.get("active") is True and "id" in data:
            active_registers.add(data["id"])

        for value in data.values():
            active_registers.update(find_activeID(value))

    elif isinstance(data, list):
        for item in data:
            active_registers.update(find_activeID(item))

    return active_registers


### Task 3

### Task 4

In [2]:
import pandas as pd


def transactions_to_dataframe(transactions):
    """Convert transactions to a chronologically sorted DataFrame."""
    df = pd.DataFrame(transactions)
    if df.empty:
        return df
    
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.set_index("timestamp").sort_index()
    return df


def hourly_revenue_by_item(df):
    """Calculate total hourly revenue for each item."""
    
    hourly_revenue = (
        df.groupby("item")["price"]
        .resample("1h")
        .sum()
    )

    return hourly_revenue


def make_pivot(df: pd.DataFrame) -> pd.DataFrame:
    """Create a pivot table showing total revenue by item and status."""

    pivot = pd.pivot_table(
        df,
        index="item",
        columns="status",
        values="price",
        aggfunc="sum",
        fill_value=0
    )

    return pivot


def refund_ratio_by_item(df: pd.DataFrame) -> pd.DataFrame:
    """Calculate the refund ratio for each item."""

    items = sorted(df["item"].unique())

    success_count = (
        df[df["status"] == "SUCCESS"]["item"]
        .value_counts()
        .reindex(items, fill_value=0)
        .to_numpy(dtype=float)
    )

    refund_count = (
        df[df["status"] == "REFUND"]["item"]
        .value_counts()
        .reindex(items, fill_value=0)
        .to_numpy(dtype=float)
    )

    refund_ratios = compute_refund_ratio(success_count, refund_count)

    return pd.DataFrame({
        "item": items,
        "refund_ratio": refund_ratios
    })

### Task 5

In [3]:
def workflow(log_path: str | Path, config_path: str | Path):
    """Run the data processing workflow and save output files."""

    # 1. Find log files
    log_files = find_log_files(log_path)
    
    print("=== Log files found ===")
    for log_file in log_files:
        print(log_file)

    # 2. Find active id
    with open(config_path, "r", encoding="utf-8") as file:
        config = json.load(file)

    active_id = find_activeID(config)

    print("\n=== Active registers ===")
    for register_id in active_id:
        print(register_id)

    # 3.


    # 4.
    df = transactions_to_dataframe(transactions)

    # 6. Create summary files
    hourly = hourly_revenue_by_item(df)
    pivot = make_pivot(df)
    refund_ratios = refund_ratio_by_item(df)

    hourly.to_json("hourly_revenue.json")
    pivot.to_csv("status_pivot_table.csv")
    refund_ratios.to_csv("item_refund_ratios.csv")

    return df, hourly, pivot, refund_ratios


df, hourly, pivot, refund_ratios = workflow(
    log_path="d2_data",
    config_path="d2_data/store_config.json")

print("\nTransactions")
print(df.head())

print("\nHourly revenue")
print(hourly.head())

print("\nStatus pivot table")
print(pivot)

print("\nRefund ratios")
print(refund_ratios)


=== Log files found ===
d2_data\2026\library_cafe\POS_01\register_sales.log
d2_data\2026\library_cafe\POS_02\register_sales.log
d2_data\2026\library_cafe\POS_03\register_sales.log
d2_data\2026\science_building\POS_01\register_sales.log
d2_data\2026\science_building\POS_02\register_sales.log
d2_data\2026\science_building\POS_03\register_sales.log
d2_data\2026\student_union\POS_01\register_sales.log
d2_data\2026\student_union\POS_02\register_sales.log
d2_data\2026\student_union\POS_03\register_sales.log

=== Active registers ===
POS_03
POS_01
POS_02


NameError: name 'transactions' is not defined